<a href="https://colab.research.google.com/github/Cassieh28/Data-analytics-portfolio/blob/main/ML_HOI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Old Dataset

In [ ]:
from pandas.plotting import scatter_matrix
from sklearn.preprocessing import LabelEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import os
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
health_data = pd.read_csv('Health_Opportunity_Index.csv')

FileNotFoundError: [Errno 2] No such file or directory: 'Health_Opportunity_Index.csv'

In [ ]:
from sklearn.model_selection import train_test_split
# Split the data into train set (80%) and test set (20%)
train_data, test_data = train_test_split(health_data, test_size=0.2, random_state=42)

In [ ]:
y_train = train_data["Health Opportunity Index"].copy()
X_train = train_data.drop(columns=["Health Opportunity Index"], axis=1)
y_test = test_data["Health Opportunity Index"].copy()
X_test = test_data.drop(columns=["Health Opportunity Index"], axis=1)

health_num = X_train.drop(columns=["Rural~Urban"], axis=1)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler,MinMaxScaler
from sklearn.impute import SimpleImputer
from sklearn.decomposition import PCA
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy="median")),
    ('std_scaler', MinMaxScaler()),
    ('pca', PCA(n_components=0.75))  # Keeps 75% of variance
])

health_num_tr = num_pipeline.fit_transform(health_num)

In [ ]:
from sklearn.compose import ColumnTransformer

num_attribs = list(health_num)
cat_attribs = ["Rural~Urban"]

full_pipeline = ColumnTransformer([
        ("num", num_pipeline, num_attribs),
        ("cat", OneHotEncoder(handle_unknown='ignore'), cat_attribs),
    ])

X_train_prepared = full_pipeline.fit_transform(X_train)
X_test_prepared = full_pipeline.transform(X_test)

In [ ]:
print(health_data.shape)
print(X_train_prepared.shape)

### Regression

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor  # Import XGBoost Regressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import numpy as np
import pandas as pd
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

# Dictionary to hold the performance metrics of the models
model_performance = {}

# Linear Regression
lr = LinearRegression()
lr.fit(X_train_prepared, y_train)
y_pred_lr = lr.predict(X_test_prepared)
mse_lr = mean_squared_error(y_test, y_pred_lr)
mae_lr = mean_absolute_error(y_test, y_pred_lr)
r2_lr = r2_score(y_test, y_pred_lr)
model_performance['Linear Regression'] = (mse_lr, mae_lr, r2_lr)

# Decision Tree Regression
dt = DecisionTreeRegressor(random_state=42)
dt.fit(X_train_prepared, y_train)
y_pred_dt = dt.predict(X_test_prepared)
mse_dt = mean_squared_error(y_test, y_pred_dt)
mae_dt = mean_absolute_error(y_test, y_pred_dt)
r2_dt = r2_score(y_test, y_pred_dt)
model_performance['Decision Tree'] = (mse_dt, mae_dt, r2_dt)

# Random Forest Regression
rf = RandomForestRegressor(random_state=42)
rf.fit(X_train_prepared, y_train)
y_pred_rf = rf.predict(X_test_prepared)
mse_rf = mean_squared_error(y_test, y_pred_rf)
mae_rf = mean_absolute_error(y_test, y_pred_rf)
r2_rf = r2_score(y_test, y_pred_rf)
model_performance['Random Forest'] = (mse_rf, mae_rf, r2_rf)

# XGBoost Regression
xgb = XGBRegressor(random_state=42)
xgb.fit(X_train_prepared, y_train)
y_pred_xgb = xgb.predict(X_test_prepared)
mse_xgb = mean_squared_error(y_test, y_pred_xgb)
mae_xgb = mean_absolute_error(y_test, y_pred_xgb)
r2_xgb = r2_score(y_test, y_pred_xgb)
model_performance['XGBoost'] = (mse_xgb, mae_xgb, r2_xgb)


# Print model performance
for model, metrics in model_performance.items():
    print(f"{model}:")
    print(f"  MSE: {metrics[0]}")
    print(f"  MAE: {metrics[1]}")
    print(f"  R^2: {metrics[2]}")
    print("------")


In [ ]:
from sklearn.model_selection import cross_val_score

lr_cv_scores = cross_val_score(lr, X_train_prepared, y_train, cv=10, scoring='r2')
print("Average 5-Fold R^2 Score: {}".format(np.mean(lr_cv_scores)))


In [ ]:
import matplotlib.pyplot as plt

residuals = y_test - y_pred_lr
plt.scatter(y_pred_lr, residuals)
plt.title('Residuals vs. Predicted Values')
plt.xlabel('Predicted Values')
plt.ylabel('Residuals')
plt.axhline(y=0, color='r', linestyle='--')
plt.show()


In [ ]:
print("Coefficients:", lr.coef_)

In [ ]:
y_train_pred = lr.predict(X_train_prepared)
mse_train_lr = mean_squared_error(y_train, y_train_pred)
r2_train_lr = r2_score(y_train, y_train_pred)

print("Train MSE:", mse_train_lr)
print("Train R^2:", r2_train_lr)


In [ ]:
# Example: Adding noise
noise = np.random.normal(0, 0.01, size=y_train.shape)
y_train_noisy = y_train + noise
lr.fit(X_train_prepared, y_train_noisy)
y_pred_noisy = lr.predict(X_test_prepared)
mse_noisy = mean_squared_error(y_test, y_pred_noisy)
r2_noisy = r2_score(y_test, y_pred_noisy)

print("Noisy MSE:", mse_noisy)
print("Noisy R^2:", r2_noisy)


In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_squared_error

# Train a new linear regression model
lr = LinearRegression()
lr.fit(X_train_prepared, y_train)

# Perform 10-fold cross-validation on the training set
lr_cv_mse_scores_train = cross_val_score(lr, X_train_prepared, y_train, cv=10, scoring='neg_mean_squared_error')

# Convert the scores to positive MSE
lr_cv_mse_scores_train = -lr_cv_mse_scores_train

# Calculate the average MSE across the 10 folds for the training data
avg_mse_cv_train = lr_cv_mse_scores_train.mean()

# Predict on the test set using the trained model
y_pred_test = lr.predict(X_test_prepared)

# Calculate the MSE for the test data
mse_test = mean_squared_error(y_test, y_pred_test)

(avg_mse_cv_train, mse_test)


In [ ]:
import matplotlib.pyplot as plt

# Plot the distribution of the target variable in the training set
plt.figure(figsize=(10, 5))
plt.hist(y_train, bins=30, alpha=0.7, color='blue', edgecolor='black')
plt.title('Distribution of the Health Opportunity Index (Training Set)')
plt.xlabel('Health Opportunity Index (HOI)')
plt.ylabel('Frequency')
plt.show()

# Plot the distribution of the target variable in the test set
plt.figure(figsize=(10, 5))
plt.hist(y_test, bins=30, alpha=0.7, color='green', edgecolor='black')
plt.title('Distribution of the Health Opportunity Index (Test Set)')
plt.xlabel('Health Opportunity Index (HOI)')
plt.ylabel('Frequency')
plt.show()

# Calculate and print the variance of the target variable in the training and test sets
train_variance = np.var(y_train)
test_variance = np.var(y_test)

print(f"Variance in Training Set: {train_variance}")
print(f"Variance in Test Set: {test_variance}")


In [ ]:
model_performance

In [ ]:
# Retrieve the intercept (β0)
intercept = lr.intercept_

# Retrieve the coefficients (β1, β2, ..., βn)
coefficients = lr.coef_

# Get the feature names if they exist or create generic feature names
# Get the feature names from the 'health' DataFrame
feature_names = X_train.columns

# Construct the linear equation as a string
linear_equation = f"y = {intercept}"
for coef, feature in zip(coefficients, feature_names):
    linear_equation += f" + ({coef}) * {feature}"

# Print the equation
print(linear_equation)


In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

# Assuming X_train_prepared is your prepared features DataFrame and has column names
feature_names = X_train_prepared.columns if 'columns' in dir(X_train_prepared) else [f'feature_{i}' for i in range(X_train_prepared.shape[1])]

plt.figure(figsize=(20, 10))
plot_tree(dt, filled=True, feature_names=feature_names, max_depth=3)
plt.title("Decision Tree Regressor")
plt.show()


In [ ]:
# Visualize a single tree from the Random Forest
plt.figure(figsize=(20, 10))
plot_tree(rf.estimators_[0], filled=True, feature_names=feature_names, max_depth=3)
plt.title("One Tree from Random Forest Regressor")
plt.show()


In [ ]:
from xgboost import plot_importance

plt.figure(figsize=(10, 8))
plot_importance(xgb, max_num_features=10)  # Show top 10 features
plt.show()


### ANN

In [ ]:
import tensorflow as tf
from tensorflow import keras


model = keras.models.Sequential([
      keras.layers.Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
      keras.layers.Dropout(0.2),
      keras.layers.Dense(64, activation='relu'),
      keras.layers.Dense(1)  # Output layer for regression
    ])

model.summary()

In [ ]:
model.compile(optimizer='nadam', loss='mean_squared_error', metrics=[tf.keras.metrics.RootMeanSquaredError()])

history = model.fit(X_train, y_train, epochs=100, validation_split=0.2)

In [ ]:
test_loss = model.evaluate(X_test, y_test)
print('Test MSE:', test_loss)

In [ ]:
from sklearn.model_selection import KFold

def build_model(input_shape):
    model = keras.models.Sequential([
        keras.layers.Dense(128, activation='relu', input_shape=(input_shape,)),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dense(1)  # Output layer for regression
    ])
    model.compile(optimizer='adam', loss='mean_squared_error', metrics=[tf.keras.metrics.MeanSquaredError()])
    return model

kfold = KFold(n_splits=5, shuffle=True, random_state=42)
fold_no = 1
mse_scores = []

for train_index, test_index in kfold.split(X):
    X_train_fold, X_test_fold = X[train_index], X[test_index]
    y_train_fold, y_test_fold = y[train_index], y[test_index]

    model = build_model(X_train_fold.shape[1])
    model.fit(X_train_fold, y_train_fold, epochs=50, batch_size=32, verbose=0)  # Adjust epochs, batch_size as needed

    scores = model.evaluate(X_test_fold, y_test_fold, verbose=0)
    print(f"Fold {fold_no}, Mean Squared Error: {scores[1]}")
    mse_scores.append(scores[1])
    fold_no += 1

average_mse = sum(mse_scores) / len(mse_scores)
print(f"Average Mean Squared Error across all folds: {average_mse}")